<a href="https://colab.research.google.com/github/E-tech-coder/DataScienceCapstoneProject/blob/Elena/DepartmentTrainingRawData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/content/sample_data/department-v2.csv").rename(columns = {"label":"department"})

In [ ]:
import random
departments = df["department"].unique().tolist()
rows = []
k = 4 # To balance positive class and negative class

for idx, row in df.iterrows():
  position = row["text"]
  true_dpt = row["department"]
  rows.append(
        {
            "premise":position,
            "hypothesis":"This job belongs to the " + str(true_dpt)+" department.",
            "labels":"entailment"
        }
    )

  neg_dpts = [d for d in departments if d!= true_dpt]
  sample_dpts = random.sample(neg_dpts, k = min(k, len(neg_dpts)))
  for neg_dpt in sample_dpts:
    rows.append(
        {
            "premise": position,
            "hypothesis": "This job belongs to the " + str(neg_dpt) +" department.",
            "labels":"contradiction"
        }
    )

NLI_df = pd.DataFrame(rows)

In [ ]:
max(NLI_df["hypothesis"].str.len())

58

In [ ]:
eval_df_raw = pd.read_csv("/content/sample_data/df_profiles_cleansed.csv")[["position", "department"]]

In [ ]:
eval_df_raw

,position,department
0,Prokurist,Other
1,CFO,Other
2,Betriebswirtin,Other
3,Prokuristin,Other
4,CFO,Other
...,...,...
2610,Justitiar,Other
2611,Geschäftsführer,Other
2612,Präsidium,Other
2613,Rechtsanwalt,Other


In [ ]:
rows=[]
for idx, row in eval_df_raw.iterrows():
  position = row["position"]
  true_dpt = row["department"]
  rows.append(
        {
            "premise":position,
            "hypothesis":"This job belongs to the " + str(true_dpt)+" department.",
            "labels":2 #entailment
        }
    )

eval_df = pd.DataFrame(rows)

In [ ]:
eval_df

,premise,hypothesis,label
0,Prokurist,This job belongs to the Other department.,2
1,CFO,This job belongs to the Other department.,2
2,Betriebswirtin,This job belongs to the Other department.,2
3,Prokuristin,This job belongs to the Other department.,2
4,CFO,This job belongs to the Other department.,2
...,...,...,...
2610,Justitiar,This job belongs to the Other department.,2
2611,Geschäftsführer,This job belongs to the Other department.,2
2612,Präsidium,This job belongs to the Other department.,2
2613,Rechtsanwalt,This job belongs to the Other department.,2


In [ ]:
mapping = {"contraction":0, "neutral":1,"entailment":2}

In [ ]:
NLI_df["label_code"] = NLI_df["labels"].map(mapping)

In [ ]:
train_df = NLI_df.drop(columns = "labels").rename(columns = {"label_code":"labels"})

In [ ]:
label2id = mapping
id2label = {k:v for v,k in label2id.items()}

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

In [ ]:
eval_dataset

Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 2615
})

In [ ]:
from transformers import AutoTokenizer

model_name = "joeddav/xlm-roberta-large-xnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize(batch):
  return tokenizer(batch["premise"],batch["hypothesis"], truncation=True, padding = "max_length", max_length = 106)

train_dataset = train_dataset.map(tokenize, batched = True)
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

eval_dataset = eval_dataset.map(tokenize, batched = True)
eval_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Map:   0%|          | 0/50725 [00:00<?, ? examples/s]

Map:   0%|          | 0/2615 [00:00<?, ? examples/s]

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
  logits,labels = eval_pred
  predictions = np.argmax(logits, axis = -1)
  return metric.compute(predictions=predictions, references=label)

In [ ]:
from transformers import Trainer, TrainingArguments
training_args = TrainingArguments("test_trainer", report_to="none")

In [ ]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = len(label2id),
    id2label=id2label,
    label2id = label2id,
    ignore_mismatched_sizes = True
)

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:

trainer = Trainer(
  model = model,
  args = training_args,
  train_dataset = train_dataset,
  eval_dataset = eval_dataset,
  compute_metrics = compute_metrics
)

trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
